
 GOLD LAYER - DIMENSION TABLE: dim_date

Goal:
 Create a date dimension to support time-based analytics:
 - Revenue trends
 - Monthly / yearly aggregation
 - Seasonality analysis
 - Weekend vs weekday behavior


1. Define Date Range

In [0]:

import datetime
start_date = datetime.date(2025, 1, 1)   # 1 Jan 2025  (d/m/y)
end_date   = datetime.date(2025, 3, 1)   # 1 Mar 2025  (d/m/y)
print(f"Min date: {start_date}, Max date: {end_date}")


2. Generate Date Sequence

In [0]:

from pyspark.sql.functions import sequence, explode
 
df_date = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{start_date}'),
        to_date('{end_date}'),
        interval 1 day
    )) AS date
""")
 
display(df_date)

3. Date Attributes (Business-Oriented)

In [0]:


from pyspark.sql.functions import (
    year, month, dayofmonth,
    dayofweek, date_format,
    weekofyear, quarter
)
 
df_date = df_date \
    .withColumn("year",        year("date")) \
    .withColumn("month",       month("date")) \
    .withColumn("month_name",  date_format("date", "MMMM")) \
    .withColumn("year_month",  date_format("date", "yyyy-MM")) \
    .withColumn("quarter",     quarter("date")) \
    .withColumn("week_of_year", weekofyear("date")) \
    .withColumn("day",         dayofmonth("date")) \
    .withColumn("day_name",    date_format("date", "EEEE")) \
    .withColumn("day_of_week", dayofweek("date"))
 
display(df_date)

 4. Business Flags (Weekend / Month Start / Month End)

In [0]:
from pyspark.sql.functions import when
 
df_date = df_date \
    .withColumn(
        "is_weekend",
        when(dayofweek("date").isin([1, 7]), True).otherwise(False)
    ) \
    .withColumn(
        "is_month_start",
        when(dayofmonth("date") == 1, True).otherwise(False)
    ) \
    .withColumn(
        "is_month_end",
        when(dayofmonth("date") >= 28, True).otherwise(False)
    )
 
display(df_date)

5. Sorting Helper

In [0]:
from pyspark.sql.functions import date_format
 
df_date = df_date.withColumn(
    "month_year_sort",
    date_format("date", "yyyyMM")
)
 
display(df_date)

6. Write to Gold Table

In [0]:


spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("DROP TABLE IF EXISTS gold.dim_date")
 
df_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_date")
 
print("✅ gold.dim_date written successfully.")

7. Validation

In [0]:

display(spark.table("gold.dim_date"))